In [ ]:
from astropy.cosmology import Planck18
#from zdm.zdm import cosmology as cos
#from zdm.zdm import misc_functions
from zdm.zdm import parameters
from zdm.zdm import figures
#from zdm import survey
from zdm.zdm import pcosmic
#from zdm import iteration as it
from zdm.zdm import loading
#from zdm import io
import matplotlib.pyplot as plt
import cmasher as cmr
import matplotlib
import pandas as pd
import numpy as np
#from pkg_resources import resource_filename
#import time
import os
import ndtest


from chime_ffff_pz.scripts import frb_table
from frb.scripts.pzdm_mag import main as pzdm_mag_main


In [ ]:




def set_state(nz):
    """
    Initializes the state with the given power-law index for redshift evolution of host DM contribution (n_z).
    """
    param_dict={'sfr_n': 0.21, 'alpha': 0.11, 'lmean': 2.18, 'lsigma': 0.42, 'lEmax': 41.37, 
                'lEmin': 39.47, 'gamma': -1.04, 'H0': 70.23, 'halo_method': 0, 'sigmaDMG': 0.0, 'sigmaHalo': 0.0,
                'lC': -7.61, 'min_lat': 0.0}
    
    state = parameters.State()
    state.set_astropy_cosmo(Planck18)
    state.update_params(param_dict)
    param_nz = {'n_z': nz}
    state.update_params(param_nz)

    # Initialize with state parameters
    #cos.set_cosmology(state.params)
    #cos.init_dist_measures()
    return state



def creategrid(state,surv):
    # creates a grid object with the given state parameters
    surveys, grids = loading.surveys_and_grids(init_state=state, survey_names = surv,   
        repeaters=False, sdir=sdir)
    
    return surveys,grids
    



def mask_plot( dmvals, zvals, state):
    """
    Plots the DM mask as a function of DM and redshift for a given state.
    """
    mask = pcosmic.get_dm_mask(dmvals, (state.host.lmean, state.host.lsigma, state.host.n_z), zvals)
    plt.imshow(mask, aspect='auto', origin='lower', extent=(dmvals[0], dmvals[-1], zvals[0], zvals[-1]))
    plt.xlabel('DM')
    plt.ylabel('Redshift')
    plt.title('DM Mask with Redshift Evolution (n_z={})'.format(state.host.n_z))
    plt.colorbar(label='Mask Value')
    plt.show()


def pzdm_plot(state,surveys,grids,axx,it,plt_styles,FRBDMs=None, FRBZs=None):
    showplot, legend, colorbar = (True, True, True) if it==3 else (False, False, False)
    ytic=True 
    legend = True if it==3 else False
    ylabel='${\\rm DM}_{\\rm EG}$' if it==0 else None # ${\\rm DM}_{\\rm cosmic} + {\\rm DM}_{\\rm host}$
    label='$\\log_{10} p({\\rm DM}_{\\rm EG},z)$ [a.u.]' if it==3 else None #$\\log_{10} p({\\rm DM}_{\\rm cosmic} + {\\rm DM}_{\\rm host},z)$ [a.u.]

    # KS test for the Macquart relation
    pval, ks_stat = sample_grid(grids[0], FRBZs, FRBDMs)
    for s, g in zip(surveys, grids):
        ######### Create a plot of p(DM,z) and show FRBs ##################
        figures.plot_grid(
                    g.rates,
                    g.zvals,
                    g.dmvals,
                    FRBDMs=FRBDMs,
                    FRBZs=FRBZs,
                    norm=3,
                    logrange=5,
                    log=True,
                    Macquart=state,
                    plt_dicts=plt_styles,
                    label=label,
                    project=False,
                    ylabel=ylabel,
                    zmax=zmax,DMmax=DMmax,
                    showplot=showplot,
                    save=False,
                    Aconts=contur,
                    cont_clrs=[1,1,1],
                    myaxes= axx,
                    it = it,
                    legend=False,
                    colorbar=colorbar,
                    ytic=ytic,
                    #title=f'$DM_{{\\rm host}}\propto (1+z)^{powind[it]}$'
                    pval=pval,
                    ks_stat=ks_stat,
                    power_host=powind[it]
                    )
        return 
    


def macquart_plot(state,surveys,grids,axx,it,plt_styles,FRBDMs=None, FRBZs=None):
    """
    Plots the Macquart relation (p(DM_cosmic + DM_host | z)) -- it is survey independent.
    """
    showplot,  colorbar = (True, True) if it==3 else (False,  False)
    legend = True if it==3 else False
    ytic=True 
    label='$\\log_{10} p({\\rm DM}_{\\rm EG}|z)$ [a.u.]' if it==3 else None
    ylabel='${\\rm DM}_{\\rm EG}$' if it==0 else None
    for s, g in zip(surveys, grids):        
        if s.zlist is not None:
            OK = s.zlist
            FRBD=s.DMEGs[s.zlist]
            FRBZ=s.Zs[s.zlist]
        else:
            FRBD=None
            FRBZ=None
    FRBDMs = np.append(FRBDMs, FRBD) if FRBD is not None else FRBD
    FRBZs = np.append(FRBZs, FRBZ) if FRBZ is not None else FRBZ
    # KS test for the Macquart relation
    #pval, ks_stat = sample_grid(grids[0], FRBZs, FRBDMs)
    for s, g in zip(surveys, grids):  
        figures.plot_grid(
                    g.smear_grid,
                    g.zvals,
                    g.dmvals,
                    norm=3,
                    log=True,
                    FRBDMs=FRBDMs,
                    FRBZs=FRBZs,
                    Macquart=state,
                    plt_dicts=plt_styles,
                    label=label,
                    project=False,
                    ylabel=ylabel,
                    zmax=zmax,DMmax=DMmax,
                        showplot=showplot,
                        save=False,
                        conts=[0.16, 0.5, 0.84 ],
                        cont_clrs=[1,1,1],
                        myaxes= axx,
                        it = it,
                        legend=legend,
                        colorbar=colorbar,
                        ytic=ytic,
                        title=f'$DM_{{\\rm host}}\propto (1+z)^{powind[it]:.0f}$',
                        pval=None,
                        ks_stat=None,
                        power_host=powind[it]
                    )
        return
        
def chime_plot(state,surveys,crates,gri,axx,it,plt_styles, FRBDMs=None, FRBZs=None):
        showplot, legend, colorbar = (True, True, True) if it==3 else (False, False, False)
        ytic=True 
        legend = True if it==3 else False
        label='$\\log_{10} p({\\rm DM}_{\\rm EG},z)$ [a.u.]' if it==3 else None #$\\log_{10} p({\\rm DM}_{\\rm cosmic} + {\\rm DM}_{\\rm host},z)$ [a.u.]
        ylabel='${\\rm DM}_{\\rm EG}$' if it==0 else None # ${\\rm DM}_{\\rm cosmic} + {\\rm DM}_{\\rm host}$
        ######### Create a plot of p(DM,z) and show FRBs ##################
        g = gri

        # KS test for the Macquart relation
        pval, ks_stat = sample_grid(g, FRBZs, FRBDMs)
        figures.plot_grid(
                    crates,
                    g.zvals,
                    g.dmvals,
                    norm=3,
                    logrange=5,
                    log=True,
                    FRBDMs=FRBDMs,
                    FRBZs=FRBZs,
                    Macquart=state,
                    plt_dicts=plt_styles,
                    label=label,
                    project=False,
                    ylabel=ylabel,
                    zmax=zmax,DMmax=DMmax,
                    showplot=showplot,
                    save=False,
                    Aconts=contur,
                    cont_clrs=[1,1,1],
                    myaxes= axx,
                    it = it,
                    legend=legend,
                    colorbar=colorbar,
                    ytic=ytic,
                    pval=pval,
                    ks_stat=ks_stat,
                    power_host=powind[it]
                    #title=f'$DM_{{\\rm host}}\propto (1+z)^{powind[it]}$'
                    )
        return 


def macquart(axes1,plt_styles):
    """ 
    Plots the Macquart relation with localised FRBs.
    """
    # I can create the subplot here maybe and then pass the axis to the plotting function?
    print (' ')
    print ('Now plotting Macquart relation with localised FRBs...')
    print (' ')
    zzt, dms = datapoints_meertrap()
    zsk,dmsk = datapoint_askap()
    zzt = np.append(zzt, zsk)
    dms = np.append(dms, dmsk)
    cz,cdm = chime_points()
    zzt = np.append(zzt, cz)
    dms = np.append(dms, cdm)
    for it,nz in enumerate(powind):
        state = set_state(nz)
        surveys,grids = creategrid(state,['DSA'])
        if it == 0:
            s = surveys[0]
            if s.zlist is not None:
                OK = s.zlist
                FRBD=s.DMEGs[s.zlist]
                FRBZ=s.Zs[s.zlist]
            else:
                FRBD=None
                FRBZ=None
            FRBDMs = np.append(dms, FRBD) if FRBD is not None else dms
            FRBZs = np.append(zzt, FRBZ) if FRBZ is not None else zzt
        # just Macquart plot
        macquart_plot(state,surveys,grids,axes1[it],it,plt_styles,FRBDMs=FRBDMs, FRBZs=FRBZs)
    for i, ax in enumerate(axes1):
        ax.tick_params(axis='y', labelleft=(i == 0))
    return

def meertrap(axes2,plt_styles):
    """ 
    Plots the MeerTRAP zDM figure with localised FRBs.
    """
    # I can create the subplot here maybe and then pass the axis to the plotting function?
    print (' ')
    print ('Now plotting MeerTRAP zDM figure with localised FRBs...')
    print (' ')
    zzt, dms = datapoints_meertrap()
    for it,nz in enumerate(powind):
        state = set_state(nz)
        surveys,grids = creategrid(state,['MeerTRAPcoherent'])
        # meertrap plot
        pzdm_plot(state,surveys,grids,axes2[it],it,plt_styles,FRBDMs=dms, FRBZs=zzt)
        

def dsa(axes3,plt_styles):
    """ 
    Plots the DSA zDM figure with localised FRBs.
    """
    print (' ')
    print ('Now plotting DSA zDM figure with localised FRBs...')
    print (' ')
    for it,nz in enumerate(powind):
        state = set_state(nz)
        surveys,grids = creategrid(state,['DSA'])
        # dsa plot
        s = surveys[0]
        if s.zlist is not None:
            OK = s.zlist
            FRBD=s.DMEGs[s.zlist]
            FRBZ=s.Zs[s.zlist]
        else:
            FRBD=None
            FRBZ=None
        pzdm_plot(state,surveys,grids,axes3[it],it,plt_styles,FRBDMs=FRBD, FRBZs=FRBZ)

        
    return 
def chime(axes4,plt_styles):
    """ 
    Plots the CHIME zDM figure with localised FRBs by compiling sums over all six declination bins.
    
    """
    # defines CHIME grids to load
    NDECBINS=6
    cnames=[]
    for i in np.arange(NDECBINS):
        cname="CHIME_decbin_"+str(i)+"_of_6"
        cnames.append(cname)
    survey_dir = '/Users/lmasriba/FRBs/zdm/zdm/data/Surveys/CHIME/'

    print (' ')
    print ('Now plotting CHIME zDM figure with localised FRBs...')
    print (' ')
    for it,nz in enumerate(powind):
        state = set_state(nz)

        css,cgs = loading.surveys_and_grids(survey_names=cnames, init_state=state, rand_DMG=False,sdir = survey_dir, repeaters=True)

        # compiles sums over all six declination bins
        crates = cgs[0].rates * 10**cgs[0].state.FRBdemo.lC * css[0].TOBS
        creps = cgs[0].exact_reps * cgs[0].state.rep.RC
        csingles = cgs[0].exact_singles * cgs[0].state.rep.RC
        
        for i,g in enumerate(cgs):
            s = css[i]
            if i ==0:
                continue
            else:
                crates += g.rates * 10**g.state.FRBdemo.lC * s.TOBS
                creps += g.exact_reps * g.state.rep.RC
                csingles += g.exact_singles * g.state.rep.RC
        
        cgs[0].rates = crates
        zx,dx = chime_points()
        chime_plot(state,css,crates,cgs[0],axes4[it],it,plt_styles,dx,zx)
    return 


def subplots():
    """
    Creates subplots for Macquart, MeerTRAP, DSA, and CHIME, and sets up plotting styles.

    """
    fig1, axes1 = plt.subplots(1, 4, figsize=(16, 4), sharey=True)   
    fig2, axes2 = plt.subplots(1, 4, figsize=(16, 4), sharey=True) 
    fig3, axes3 = plt.subplots(1, 4, figsize=(16, 4), sharey=True) 
    fig4, axes4 = plt.subplots(1, 4, figsize=(16, 4), sharey=True) 

    fig1.suptitle('Macquart',fontsize=14)
    fig2.suptitle('MeerTRAP',fontsize=14)
    fig3.suptitle('DSA-110',fontsize=14)
    fig4.suptitle('CHIME',fontsize=14)
 
    defaultsize=14
    ds=4
    font = {
            # 'family' : 'Helvetica',
            'weight' : 'normal',
            'size'   : defaultsize}
    matplotlib.rc('font', **font)

    # Set colours and styles for plotting contours and FRBs
    for ax in [axes1, axes2, axes3, axes4]:
        for a in ax:
            a.yaxis.set_tick_params(labelleft=True)
    data_clrs = 'k'
    markers="x"
    markersize = 5
    ewidths = 1

    plt_styles = {
        'color': data_clrs,
        'marker': markers,
        'markersize': markersize,
        'label': None,
        'markeredgewidth': ewidths
    }
    return  axes1,  axes2, axes3,  axes4, plt_styles, fig1, fig2, fig3, fig4

def datapoints_meertrap():
    meerkat_df = pd.read_csv('/Users/lmasriba/FRBs/zdm/papers/lsst/Data/meerkat_mr.txt', delim_whitespace=True,  comment='#')
    meerkat_df.columns = ['name',  'DMX', 'zspec', 'RA', 'Dec',  'zspec_err']
    zzt = np.array(meerkat_df['zspec'].values,dtype=np.float64)
    dms = np.array(meerkat_df['DMX'].values,dtype=np.float64)
    
    zzt = np.append(zzt, 2.148)
    dms = np.append(dms, 2398.03)


    return zzt, dms

def datapoint_askap():
    return 1.016, 1427


def read_ffffpz_table():

    """
    Read the FRB table from FFFF-PZ.

    Returns:
        pd.DataFrame: DataFrame containing the FRB table.
    """
    
    # Create an instance of the FRBTable class
    response = frb_table.request_frb_table()
    # Process the response to get the DataFrame
    if response is None:
        raise ValueError("No data received from the FRB table request. Please check the request parameters or the service status.")
    tbl = frb_table.process_frb_table(response)
    if tbl is None or tbl.empty:
        raise ValueError("The FRB table is empty or could not be processed. Please check the data source.")
    return tbl

def chime_points():
    """Extracts the DM and redshift values for CHIME FRBs from the FFFF-PZ table."""
    tbl = pd.read_csv('/Users/lmasriba/FRBs/zdm/papers/hostz/chimetable.csv')
    # select only CHIME FRBs with column 'status' = 'Redshift',POx' >= 0.9, and 'z' >= 0.0
    chime_frbs = tbl[(tbl['status'] == 'Redshift') & (tbl['POx'] >= 0.9) & (tbl['z'] >= 0.0)]
    chime_dms = chime_frbs['DM'].values - chime_frbs['DM_ISM'].values
    chime_zs = chime_frbs['z'].values
    chime_dms = np.append(chime_dms,1490.)
    chime_zs = np.append(chime_zs, 1.24)
    return chime_zs, chime_dms


def sample_grid(gri,z_data, dm_data):

    pvals = np.empty(boots,dtype=np.float64)
    ks_stats = np.empty(boots,dtype=np.float64)

    for i in range(boots):
        # create DM_EG by sampling the grid
        frbs = gri.GenMCSample(Nsample)
        frbs = np.array(frbs)
        z_sample = frbs[:,0]
        dm_sample = frbs[:,1]

        # perform 2D KS test between the sampled points and the data points
        P = ndtest.ks2d2s(z_data, dm_data, z_sample, dm_sample, extra=False)

        pvals[i] = P
        #ks_stats[i] = D

    return np.median(pvals), 1. # np.median(ks_stats)

    
def bigplot():
    powns = [0.0, 2.0, 3.0]
    colormap = ['Blues', 'Oranges', 'Greens']
    cont_color = ['DeepSkyBlue', 'orange', 'g']
    alphas = [0.1, 0.3, 0.15]

    fig, axo = plt.subplots(figsize=(6, 4))

    defaultsize=12
    ds=4
    font = {
            # 'family' : 'Helvetica',
            'weight' : 'normal',
            'size'   : defaultsize}
    matplotlib.rc('font', **font)



    # plot just the points here to get the legend right, then plot the contours in the loop
    state = set_state(0)
    names=["MeerTRAPcoherent","DSA","CRAFT_ICS_1300"]
    ss,gs = loading.surveys_and_grids(
            survey_names=names,repeaters=False,init_state=state,sdir=sdir,verbose=False)
        
    s=ss[0]
    g=gs[0]
    name = names[0]

        
    ###### Get list of z and dm for DSA, CRAFT and CHIME localised FRBs #####
    ICS_names=["CRAFT_ICS_892", "CRAFT_ICS_1632"]
    ics_ss, ics_gs = loading.surveys_and_grids(survey_names=ICS_names, sdir=sdir, init_state=state, verbose=False)

    dsa_Zs = ss[1].Zs[ss[1].zlist]
    dsa_DMs = ss[1].DMEGs[ss[1].zlist]

    ics_Zs = np.array([ss[2].Zs[ss[2].zlist].tolist() + ics_ss[0].Zs[ics_ss[0].zlist].tolist() + ics_ss[1].Zs[ics_ss[1].zlist].tolist()])
    ics_DMs = np.array([ss[2].DMEGs[ss[2].zlist].tolist() + ics_ss[0].DMEGs[ics_ss[0].zlist].tolist() + ics_ss[1].DMEGs[ics_ss[1].zlist].tolist()])

    meerkat_df = pd.read_csv('/Users/lmasriba/FRBs/zdm/papers/lsst/Data/meerkat_mr_all.txt', delim_whitespace=True, comment='#')
    meerkat_df.columns = ['name', 'DMX', 'zspec', 'RA', 'Dec', 'zspec_err']
    zzt = np.array(meerkat_df['zspec'].values, dtype=np.float64)
    dms = np.array(meerkat_df['DMX'].values, dtype=np.float64)

    zzt = np.append(zzt, 2.148)
    dms = np.append(dms, 2398.03)

    ###### plots MeerTRAP zDM figure ###########
    Zs = [zzt, dsa_Zs, None, ics_Zs]
    DMs = [dms, dsa_DMs, None, ics_DMs]
    point_labels = [None, None, None, None]

    data_clrs = ['r', 'b', 'k', '0.5']
    markers = ["s", ">", "o", "+"]
    markersize = [6, 6, 4, 8]
    ewidths = [0.1, 0.1, 0.1, 0.1]
    alphax = [0.7, 0.5, 1, 1]

    plt_dicts = []
    cont_dicts = None
    for i in range(len(data_clrs)):
        plt_styles = {
            'color': data_clrs[i],
            'marker': markers[i],
            'markersize': markersize[i],
            'label': point_labels[i],
            'markeredgewidth': ewidths[i],
            'alpha': alphax[i]
        }
        plt_dicts.append(plt_styles)




    for it,nz in enumerate(powns):

        showplot,  colorbar = (True, True) if it==2 else (False,  False)
        legend = True if it==2 else False
        ytic=True 
        label='$\\log_{10} p({\\rm DM}_{\\rm EG}|z)$ [a.u.]' if it==2 else None
        ylabel='${\\rm DM}_{\\rm EG}$' if it==2 else None

        state = set_state(nz)
        names=["MeerTRAPcoherent","DSA","CRAFT_ICS_1300"]
        ss,gs = loading.surveys_and_grids(
            survey_names=names,repeaters=False,init_state=state,sdir=sdir)
        
        s=ss[0]
        g=gs[0]
        name = names[0]


        
        figures.plot_grid(
                    g.smear_grid,
                    g.zvals,
                    g.dmvals,
                    norm=3,
                    log=True,
                    FRBDMs=DMs,
                    FRBZs=Zs,
                    plt_dicts=plt_dicts,
                    label=label,
                    project=False,
                    ylabel=ylabel,
                    zmax=zmax,DMmax=5000.,
                        showplot=showplot,
                        save=False,
                        conts=[0.16, 0.5, 0.84],
                        cont_clrs=np.repeat(cont_color[it],3),
                        myaxes= axo,
                        it = it,
                        legend=legend,
                        colorbar=colorbar,
                        ytic=ytic,
                        #title=f'$DM_{{\\rm host}}\propto (1+z)^{powind[it]:.0f}$',
                        pval=None,
                        ks_stat=None,
                        power_host=nz,
                        cmap = None,
                        alpha=alphas[it],
                        inbetween=True
                    )

   

    


In [ ]:
# parameters
sdir = '/Users/lmasriba/FRBs/zdm/zdm/data/Surveys'

# power-law index for redshift evolution of host DM contribution. DM_host(z) = DM_host(z=0) * (1+z)^n_z.
powind = np.array([0., 1., 2., 3.])
zvals = np.linspace(0.0, 5.0, 500)
dmvals = np.linspace(0.0, 5000.0, 1000)
contur = [1. - 0.99, 1. - 0.86, 1. - 0.39]

zmax=4
DMmax=5000
Nsample = 1000
boots = 100

2d ks test and degeneracies with params with fisher formalism or nested sampling

In [ ]:
def main():

    bigplot()
    lkakjsda

    axes1, axes2, axes3, axes4, plt_styles, fig1, fig2, fig3, fig4 = subplots()
    macquart(axes1,plt_styles)
    meertrap(axes2,plt_styles)
    dsa(axes3,plt_styles)
    chime(axes4,plt_styles)
    
   
    
main()
